# Introduction to Simple RAG Pipeline

### This notebook demonstrates a basic Retrieval-Augmented Generation (RAG) pipeline

### using HuggingFace models for embeddings and LLM, FAISS for vector storage and LangChain for orchestration. We will use a simple text file as our knowledge base.

In [1]:
# 1. Setup: Import necessary libraries and define paths
!pip install -q langchain_community
import os
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

## Load data

Load the data from a text file.


In [2]:
# 2. Load Data: Load the text file
file_path = "knowledge_base.txt"
with open(file_path, "w") as f:
    f.write("The quick brown fox jumps over the lazy dog.\n")
    f.write("Retrieval-Augmented Generation (RAG) is a natural language processing framework.\n")
    f.write("It combines the strengths of retrieval-based and generation-based AI models.\n")
    f.write("FAISS is a library for efficient similarity search and clustering of dense vectors.")

loader = TextLoader(file_path)
documents = loader.load()
print(f"Loaded {len(documents)} document(s).")
print(documents[0].page_content[:100])

Loaded 1 document(s).
The quick brown fox jumps over the lazy dog.
Retrieval-Augmented Generation (RAG) is a natural langu


## Split text
Split the loaded text into smaller chunks.


In [3]:
# 3. Split Text: Split the loaded text into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
text_chunks = text_splitter.split_documents(documents)

print(f"Created {len(text_chunks)} text chunks.")
print("First text chunk:")
print(text_chunks[0].page_content)

Created 4 text chunks.
First text chunk:
The quick brown fox jumps over the lazy dog.


## Create embeddings
Generate embeddings for the text chunks using a HuggingFace model.


In [4]:
# 4. Create Embeddings: Generate embeddings for the text chunks
embeddings = HuggingFaceEmbeddings()
text_chunk_embeddings = embeddings.embed_documents([chunk.page_content for chunk in text_chunks])

print(f"Generated embeddings for {len(text_chunk_embeddings)} text chunks.")
print("First text chunk embedding (first 10 dimensions):")
print(text_chunk_embeddings[0][:10])

README.md: 0.00B [00:00, ?B/s]

Generated embeddings for 4 text chunks.
First text chunk embedding (first 10 dimensions):
[-0.03429264575242996, -0.0013393532717600465, 0.004336133133620024, -0.0018684039823710918, 0.025440845638513565, 0.028018992394208908, -0.009773620404303074, 0.022119024768471718, -0.028821833431720734, -0.046994082629680634]


## Build vector store
Create a FAISS vector store from the embeddings.


In [5]:
!pip install -q faiss-cpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [6]:
# 5. Create Vector Store: Create a FAISS vector store from the embeddings
vector_store = FAISS.from_documents(text_chunks, embeddings)
print("FAISS vector store successfully created.")

FAISS vector store successfully created.


## Initialize llm
Load a HuggingFace language model.


In [18]:
# 6. Load Language Model: Load a HuggingFace language model
model_name = "distilbert/distilgpt2" # Choose a suitable small model
generator = pipeline('text-generation', model=model_name,max_new_tokens=150)
llm = HuggingFacePipeline(pipeline=generator)

print(f"HuggingFace language model '{model_name}' loaded successfully.")

Device set to use cpu


HuggingFace language model 'distilbert/distilgpt2' loaded successfully.


In [19]:
# Ask a question without RAG
query_without_rag = "What is RAG?"
answer_without_rag = llm.invoke(query_without_rag)

print("Query (without RAG):", query_without_rag)
print("Answer (without RAG):", answer_without_rag)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Query (without RAG): What is RAG?
Answer (without RAG):  And what is a group like RAG? For me, this is RAG. I think it is an important part of the game and what RAG was all about was what I thought of it. And it was a lot of fun to play with it. You are basically playing as you would have any other role in the game so that you can do something even more interesting. So, it was fun to play with it and to do something amazing and interesting. So, the game, with a lot of detail, like its design and mechanics as well as the level design from the perspective of a gamer, it turned out that the game was a bit crazy for me to explore even more than that.



Yeah, like there


## Setup rag chain
Create a RetrievalQA chain using the LLM and vector store.


In [20]:
# 7. Create RetrievalQA Chain: Create a RetrievalQA chain using the LLM and vector store
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever()
)

print("RAG chain (RetrievalQA) set up successfully.")

RAG chain (RetrievalQA) set up successfully.


## Query rag chain

Ask a question to the RAG chain and get an answer.


In [21]:
# 8. Ask a question: Ask a question to the RAG chain
query = "What is RAG?"
answer = qa_chain.invoke(query)

# Print the answer
print("Query:", query)
print("Answer:", answer)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Query: What is RAG?
Answer: {'query': 'What is RAG?', 'result': ' "RAG is a network of data structures based on a variety of network structures and their underlying rules. RAG represents a kind of network. It includes a set of common features, including a set of common features, such as the types of algorithms, such as a shared data structure, the network of algorithms, and rules. It includes a network of data structures called RAG".\nOne of RAG\'s main features is how it calculates the data that each member of the RAG group has.\nThe RAG network takes a deep-learning network and stores the data in an RAG database:\nA typical RAG block is:\nA small set of features with lots of different features:\nA simple set of features with lots'}
